# 실습 2: 퍼셉트론을 쌓아 신경망 만들기

## 오늘 할 일 — 75분

이론 2장의 **신경망 그림을 PyTorch 코드로 옮긴다.**
입력에서 출력까지 한 층씩 계산하고, 연결되는 텐서의 모양과 파라미터 수를 확인한다.
오늘은 가중치가 주어진 모형의 출력을 계산한다. 가중치를 학습하는 코드는 **4주차 실습 3**에서 작성한다.

- **대응 이론**: [Ch02 퍼셉트론과 다층 퍼셉트론](../chapters/ch02.qmd)
- **준비 지식**: 실습 1의 텐서 생성, 인덱싱, `shape`, `@`.
- **실행 환경**: Google Colab의 기본 CPU 런타임. 외부 데이터 다운로드는 필요 없다.
- 직접 해보기에서는 2~4분 멈추어 작성한다. 실습은 제출하거나 채점하지 않는다.
- `None`을 채우고 확인 코드를 실행한다. 힌트는 문제 아래, 정답은 문서 끝의 **해설**에 있다.

| 시간 | 내용 |
|---|---|
| 0–8분 | 입력 준비와 `layer(X)` |
| 8–23분 | 이론의 가중합과 출력 확인 |
| 23–35분 | 활성화 함수 |
| 35–50분 | 두 층을 직접 연결하고 오류 수정 |
| 50–63분 | `nn.Sequential`과 파라미터 수 |
| 63–75분 | 종합 연습과 정리 |

## 1. 입력을 층에 넣기 — 8분

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X = torch.tensor([[0., 0.],
                  [1., 0.],
                  [0., 1.],
                  [1., 1.]])
layer = nn.Linear(2, 1)
z = layer(X)
print(X.shape, z.shape)        # (4, 2) → (4, 1)
print(z)

`nn`에는 신경망 부품이 들어 있다. `manual_seed`는 무작위 초기값을 재현하기 위한 설정이다.
이번 실습에서는 다음 세 줄의 사용법을 반복해서 익힌다.

| 코드 | 의미 |
|---|---|
| `X = torch.tensor(...)` | 한 행에 데이터 한 건, 한 열에 입력 변수 하나 |
| `layer = nn.Linear(2, 1)` | 입력 변수 2개로 출력 값 1개를 만드는 층 생성 |
| `z = layer(X)` | 입력을 넣어 출력 계산 |

입력은 `(데이터 수, 입력 변수 수)`, 출력은 `(데이터 수, 출력 변수 수)`다.
한 건을 계산할 때도 **한 행짜리 입력**을 넣는다.

In [ ]:
one_X = X[:1]
one_z = layer(one_X)
print(one_X.shape, one_z.shape)   # (1, 2) → (1, 1)

**확인 질문**: 사원이 4명에서 8명으로 늘면 `nn.Linear(2, 1)`의 숫자를 바꿔야 하는가?

## 2. 이론의 가중합과 맞춰 보기 — 15분

`nn.Linear`는 가중합을 계산한다. 활성화 함수는 다음 절에서 붙인다.
처음 만들면 가중치와 편향이 초기화되어 있다.

In [ ]:
print(layer.weight)        # 출력 하나에 연결된 가중치 두 개
print(layer.bias)          # 출력 하나의 편향 한 개

이론의 퍼셉트론은 $z=0.5x_1+0.5x_2-0.3$이다.
아래 셀은 이론의 숫자를 넣는 **제공 코드**다. 그대로 실행한다.
`no_grad()`는 기울기 기록 없이 값을 설정하는 구간이며, `copy_()`는 기존 파라미터에 값을 복사한다.
이 설정 코드를 외울 필요는 없다. 이후 학습 실습에서는 가중치를 자동으로 조정한다.

In [ ]:
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[0.5, 0.5]]))
    layer.bias.copy_(torch.tensor([-0.3]))

z = layer(X)
print(z)

두 번째 데이터 $(1,0)$의 값은 $0.5\times1+0.5\times0-0.3=0.2$다.
나머지 세 값도 이론의 식으로 확인한다. **코드에서 출력은 계속 `layer(X)`로 계산한다.**

In [ ]:
expected_z = torch.tensor([[-0.3], [0.2], [0.2], [0.7]])
print(torch.allclose(z, expected_z))

`torch.allclose`는 소수 계산의 작은 오차를 허용하며 값이 같은지 확인한다.
파라미터는 출력별로 묶여 있다. 출력이 1개여도 다음 규칙을 그대로 쓴다.

| 대상 | 모양 | `nn.Linear(2, 1)`의 경우 |
|---|---|---|
| `layer.weight` | `(출력 수, 입력 수)` | `(1, 2)` |
| `layer.bias` | `(출력 수,)` | `(1,)` |
| `layer(X)` | `(데이터 수, 출력 수)` | `(4, 1)` |

### 직접 해보기 ① — 입력 3개, 출력 4개

입력 데이터 5건을 처리하는 층을 만들고 출력과 가중치 모양을 확인한다.
가중치는 초기값 그대로 사용한다.

In [ ]:
input5 = torch.tensor([[1., 2., 3.],
                       [2., 3., 4.],
                       [3., 4., 5.],
                       [4., 5., 6.],
                       [5., 6., 7.]])

In [ ]:
# ✏️ 직접 채워 보세요
practice_layer = None
practice_out = None

assert isinstance(practice_layer, nn.Linear)
assert practice_layer.in_features == 3 and practice_layer.out_features == 4
assert practice_layer.weight.shape == (4, 3)
assert practice_layer.bias.shape == (4,)
assert practice_out is not None and practice_out.shape == (5, 4)
assert torch.allclose(practice_out, practice_layer(input5))
print('통과')

힌트: `nn.Linear`의 두 숫자는 **변수 수와 출력 수**다. 데이터 건수 5는 넣지 않는다.

## 3. 활성화 함수 붙이기 — 12분

이론에서는 같은 가중합에 항등 함수를 쓰면 회귀값을, 계단 함수를 쓰면 0 또는 1을 얻었다.

In [ ]:
print(z)                   # 항등 함수: 그대로 출력
print((z >= 0).float())    # 이론의 계단 함수: 조건 비교를 0/1로 변환

계단 함수는 이론의 계산을 확인하기 위한 것이다. 이후 학습할 신경망에는 ReLU나 Sigmoid 등의 부품을 사용한다.

### 부품 생성과 호출을 두 줄로 쓰기

In [ ]:
values = torch.tensor([[-2.], [-0.5], [0.], [1.5], [3.]])
relu = nn.ReLU()                  # 부품 생성
activated = relu(values)           # 텐서에 적용
print(activated)
print(values.shape, activated.shape)

In [ ]:
sigmoid = nn.Sigmoid()
tanh = nn.Tanh()
print(sigmoid(values))
print(tanh(values))

ReLU는 음수를 0으로 바꾸고, Sigmoid는 값을 0과 1 사이로, Tanh는 -1과 1 사이로 바꾼다.
세 함수 모두 **입력의 모양을 유지**한다.

### 직접 해보기 ② — 출력 예상 후 함수 적용

`probe`의 ReLU 출력값을 먼저 적는다. 그다음 ReLU와 Sigmoid를 적용한다.

In [ ]:
probe = torch.tensor([[-3., 2.],
                      [0., -1.]])

In [ ]:
# ✏️ 직접 채워 보세요
expected_relu = None       # 예상한 결과를 torch.tensor로 작성
relu_result = None         # ReLU 부품을 적용
sigmoid_result = None      # Sigmoid 부품을 적용

assert expected_relu is not None and expected_relu.shape == probe.shape
assert torch.equal(expected_relu, torch.tensor([[0., 2.], [0., 0.]]))
assert relu_result is not None and torch.equal(relu_result, expected_relu)
assert sigmoid_result is not None and sigmoid_result.shape == probe.shape
assert torch.allclose(sigmoid_result, torch.sigmoid(probe))
print('통과')

힌트: 위에서 만든 `relu`와 `sigmoid`에 `probe`를 넣는다.

## 4. 두 층을 한 줄씩 연결하기 — 15분

이론의 **입력 3 → 은닉 4 → 출력 1** 구조를 만든다. 입력은 `input5`를 그대로 쓴다.

In [ ]:
hidden_layer = nn.Linear(3, 4)
hidden_activation = nn.ReLU()
output_layer = nn.Linear(4, 1)

z1 = hidden_layer(input5)
h1 = hidden_activation(z1)
out = output_layer(h1)

print('입력:', input5.shape)
print('은닉 가중합:', z1.shape)
print('은닉 활성화:', h1.shape)
print('출력:', out.shape)

앞 층의 출력 수가 다음 층의 입력 수와 같아야 한다.
여기서는 **회귀값 하나를 출력**하므로 마지막 `Linear` 뒤에는 활성화를 추가하지 않는다.
초기 가중치로 계산했으므로 출력값이 정답에 가까울 것이라고 기대하지는 않는다.

### 직접 해보기 ③ — 연결 오류 고치기

다음 설계는 은닉층의 출력이 6개인데 출력층은 4개를 받도록 되어 있다.
입력 3 → 은닉 6 → 출력 1이 되도록 `fixed_output`을 만들고 끝까지 계산한다.

In [ ]:
wide_hidden = nn.Linear(3, 6)
bad_output = nn.Linear(4, 1)
wide_relu = nn.ReLU()
wide_h = wide_relu(wide_hidden(input5))
print('넘기는 값:', wide_h.shape)
print('현재 출력층이 받는 변수 수:', bad_output.in_features)
# bad_output(wide_h)는 모양이 맞지 않아 에러가 난다.

In [ ]:
# ✏️ 직접 채워 보세요
fixed_output = None
fixed_pred = None

assert isinstance(fixed_output, nn.Linear)
assert fixed_output.in_features == 6 and fixed_output.out_features == 1
assert fixed_pred is not None and fixed_pred.shape == (5, 1)
assert torch.allclose(fixed_pred, fixed_output(wide_h))
print('통과')

힌트: 데이터 건수는 그대로 5다. 바뀌어야 하는 것은 다음 층이 받는 **변수 수**다.

### 다음 층으로 넘어가는 값을 확인한다

In [ ]:
print(z1[:2])          # 첫 두 데이터의 은닉 가중합
print(h1[:2])          # ReLU를 통과한 값
print(out[:2])         # 출력층을 통과한 값

이론에서는 비선형 활성화가 있어야 층을 쌓는 효과가 생긴다는 것을 배웠다.
여기서는 ReLU가 음수 값을 0으로 바꾸고, 그 결과를 다음 층이 받는지 확인한다.
**확인 질문**: ReLU 전후에 값은 어떻게 바뀌었는가? 행과 열의 수는 바뀌었는가?

---

## 5. 같은 구조를 `nn.Sequential`로 묶기 — 13분

이미 만든 부품을 순서대로 묶는다. **같은 부품, 같은 파라미터**를 쓰므로 앞에서 계산한 값과 같다.

In [ ]:
model = nn.Sequential(
    hidden_layer,
    hidden_activation,
    output_layer,
)
print(model)
print(torch.allclose(model(input5), out))

보통은 아래처럼 생성하면서 바로 묶는다. 새 `Linear`를 만들면 파라미터도 새로 초기화되므로,
구조가 같아도 출력값까지 같지는 않다.

In [ ]:
new_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)
print(new_model(input5).shape)
print(new_model[0].weight.shape)    # 첫 부품에 인덱스로 접근

### 반복문으로 모양 추적하기

`for`는 부품을 앞에서부터 하나씩 꺼낸다. `h`에 직전 부품의 출력을 담아 다음 부품에 넘긴다.

In [ ]:
h = input5
for part in new_model:
    h = part(h)
    print(part, '→', h.shape)

### 파라미터 수 확인하기

이론의 계산은 $(3\times4+4)+(4\times1+1)=21$개다. ReLU에는 학습할 파라미터가 없다.
`numel()`은 텐서의 전체 원소 수를 반환한다.

In [ ]:
print(new_model[0].weight.numel())    # 12
print(new_model[0].bias.numel())      # 4

parameter_count = 0
for parameter in new_model.parameters():
    parameter_count = parameter_count + parameter.numel()
print('전체 파라미터:', parameter_count)

### 직접 해보기 ④ — 그림을 코드로 옮기기

**입력 3 → 은닉 2(ReLU) → 출력 1(항등)**인 모델을 만든다.
파라미터 수를 먼저 예상한 뒤 반복문으로 확인한다.

In [ ]:
# ✏️ 직접 채워 보세요
small_model = None
expected_count = None

assert isinstance(small_model, nn.Sequential)
assert len(small_model) == 3
assert isinstance(small_model[0], nn.Linear) and isinstance(small_model[1], nn.ReLU)
assert isinstance(small_model[2], nn.Linear)
assert small_model[0].in_features == 3 and small_model[0].out_features == 2
assert small_model[2].in_features == 2 and small_model[2].out_features == 1
assert small_model(input5).shape == (5, 1)
actual_count = 0
for parameter in small_model.parameters():
    actual_count = actual_count + parameter.numel()
assert actual_count == 11 and expected_count == actual_count
print('통과')

힌트: 각 `Linear`마다 **입력 수 × 출력 수 + 출력 수**를 더한다.

## 6. 종합 연습 — 12분

이론의 더 깊은 구조를 코드로 옮긴다.

**입력 3 → 은닉 4(ReLU) → 은닉 4(ReLU) → 출력 2(항등)**

출력 두 개는 두 개의 실수 예측값으로 생각한다. 분류 확률로 해석하는 실습은 뒤에서 다룬다.

1. `deep_model`을 만든다.
2. 입력 5건의 출력 `deep_pred`를 구한다.
3. 파라미터 수를 예상하여 `deep_expected_count`에 적는다.
4. 첫 데이터 **한 건만** 2차원을 유지해서 넣고 `one_pred`를 구한다.

In [ ]:
# ✏️ 직접 채워 보세요
deep_model = None
deep_pred = None
deep_expected_count = None
one_pred = None

assert isinstance(deep_model, nn.Sequential) and len(deep_model) == 5
assert isinstance(deep_model[0], nn.Linear) and isinstance(deep_model[1], nn.ReLU)
assert isinstance(deep_model[2], nn.Linear) and isinstance(deep_model[3], nn.ReLU)
assert isinstance(deep_model[4], nn.Linear)
assert (deep_model[0].in_features, deep_model[0].out_features) == (3, 4)
assert (deep_model[2].in_features, deep_model[2].out_features) == (4, 4)
assert (deep_model[4].in_features, deep_model[4].out_features) == (4, 2)
assert deep_pred is not None and deep_pred.shape == (5, 2)
assert torch.allclose(deep_pred, deep_model(input5))
deep_actual_count = 0
for parameter in deep_model.parameters():
    deep_actual_count = deep_actual_count + parameter.numel()
assert deep_actual_count == 46 and deep_expected_count == deep_actual_count
assert one_pred is not None and one_pred.shape == (1, 2)
assert torch.allclose(one_pred, deep_pred[:1], atol=1e-6)
print('통과')

먼저 끝났다면 **첫 은닉층만 4→6개로** 바꾸어 새 모델을 만든다.
어느 `Linear` 두 곳을 고쳐야 하는가? 출력 모양은 같은가? 파라미터 수는 몇 개 늘었는가?

## 이번 주에 손에 익힐 문법

| 할 일 | 코드 |
|---|---|
| 가중합 부품 생성 | `nn.Linear(입력 수, 출력 수)` |
| 파라미터 확인 | `layer.weight`, `layer.bias` |
| 부품으로 계산 | `layer(X)` |
| 활성화 적용 | `activation = nn.ReLU()`, `activation(z)` |
| 순서대로 연결 | `nn.Sequential(...)` |
| 특정 부품 선택 | `model[0]` |
| 층별 계산 확인 | `for part in model:` |
| 파라미터 수 확인 | `model.parameters()`, `parameter.numel()` |

**마무리 질문**: 입력 데이터가 5건에서 10건으로 늘면 `nn.Linear`를 바꿔야 하는가?
입력 변수가 3개에서 5개로 늘면 어디를 바꿔야 하는가?

3주차에는 이론 3장의 손실 계산을 손으로 연습한다.
**4주차** [실습 3: 학습 루프를 직접 만든다](lab03.qmd)에서는 오늘 조립한 모형의 가중치를 자동으로 바꾸는 코드를 배운다.

---

## 선택 참고 — `nn.Linear` 내부의 행렬 곱

수업의 필수 연습은 위에서 끝난다. 실습 1의 `@`와 연결이 궁금할 때 확인한다.
PyTorch의 `weight`는 `(출력 수, 입력 수)`로 저장된다. 2차원 행렬의 `.T`는 행과 열을 바꾼다.
따라서 이 실습의 2차원 입력에서 다음 두 계산은 같다.

```python
z = layer(X)
z_check = X @ layer.weight.T + layer.bias
print(torch.allclose(z, z_check))
```

모델을 만들고 사용하는 실습에서는 `layer(X)`를 쓴다.
저장 모양과 계산식은 [PyTorch의 Linear 문서](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)에서도 확인할 수 있다.

---

## 해설 — 먼저 직접 풀고 확인하기

해설 코드를 해당 문제 셀로 옮긴 뒤 그 문제의 확인 코드로 검산한다.

### ① 여러 출력을 만드는 층

```python
practice_layer = nn.Linear(3, 4)
practice_out = practice_layer(input5)
```

### ② 활성화

```python
expected_relu = torch.tensor([[0., 2.], [0., 0.]])
relu_result = relu(probe)
sigmoid_result = sigmoid(probe)
```

### ③ 연결 수정

```python
fixed_output = nn.Linear(6, 1)
fixed_pred = fixed_output(wide_h)
```

### ④ 작은 모델

```python
small_model = nn.Sequential(
    nn.Linear(3, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
)
expected_count = (3 * 2 + 2) + (2 * 1 + 1)
```

### 종합 연습

```python
deep_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
)
deep_pred = deep_model(input5)
deep_expected_count = (3 * 4 + 4) + (4 * 4 + 4) + (4 * 2 + 2)
one_pred = deep_model(input5[:1])
```

추가 연습에서는 첫 층을 `nn.Linear(3, 6)`, 두 번째 가중합 층을 `nn.Linear(6, 4)`로 바꾼다.
파라미터 수는 $24+28+10=62$개로 16개 늘고, 출력 모양은 `(5, 2)`로 같다.
데이터 건수는 `Linear`의 설정에 들어가지 않지만, 입력 변수 수가 바뀌면 첫 `Linear`의 입력 수를 바꿔야 한다.